In [ ]:
!pip install frechet_audio_distance panns_inference librosa pandas numpy scipy torch soundfile -q
!pip install laion-clap

In [ ]:
#-----------------------------------------------------------
#  pkg requirements
# !pip install frechet_audio_distance panns_inference librosa pandas numpy scipy torch soundfile -q
# !pip install laion-clap
#-----------------------------------------------------------

import os
import glob
import numpy as np
import pandas as pd
import torch
import librosa
import scipy
import tempfile
import soundfile as sf

from scipy.stats import kurtosis
from frechet_audio_distance import FrechetAudioDistance
from panns_inference import AudioTagging

# =========================================================
# PATHS
# =========================================================
ORIGINAL_DIR = "./original"
GENERATED_DIR = "./generated"

EVAL_PAIRS = [
    (
        "Original vs Generated",
        ORIGINAL_DIR,
        GENERATED_DIR,
    ),
]

# =========================================================
# HELPERS
# =========================================================
def safe_div(a, b, eps=1e-8):
    return a / (b + eps)

def normalize_score(val, ref=1.0):
    if np.isnan(val) or np.isinf(val):
        return 0.0
    val = abs(val)
    return float(np.exp(-val / (ref + 1e-8)))

def traj_similarity(x, y, sr=22050, n_mfcc=20):
    mfcc_x = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=n_mfcc).T
    mfcc_y = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T

    if len(mfcc_x) == 0 or len(mfcc_y) == 0:
        return 0.0

    dist_matrix = scipy.spatial.distance.cdist(mfcc_x, mfcc_y, metric='euclidean')
    D, wp = librosa.sequence.dtw(C=dist_matrix.T)
    d = D[-1, -1] / max(len(wp), 1)
    return normalize_score(d, ref=50.0)

def cycle_similarity(x, y, sr=22050):
    E_x = librosa.feature.rms(y=x)[0]
    E_y = librosa.feature.rms(y=y)[0]

    min_len = min(len(E_x), len(E_y))
    if min_len < 2:
        return 0.0

    E_x = E_x[:min_len]
    E_y = E_y[:min_len]

    mid = min_len // 2
    if mid == 0:
        return 0.0

    R_x = safe_div(np.sum(E_x[:mid]), np.sum(E_x[mid:]))
    R_y = safe_div(np.sum(E_y[:mid]), np.sum(E_y[mid:]))

    diff_ratio = abs(R_x - R_y)
    max_ratio = max(R_x, R_y, 1e-8)
    S_R = max(0, 1 - (diff_ratio / max_ratio))

    diff_env = np.sum(np.abs(E_x - E_y))
    sum_env = np.sum(E_x) + 1e-8
    S_E = max(0, 1 - (diff_env / sum_env))

    return float((S_R + S_E) / 2)

def wheeze_similarity(x, y, sr=22050, fmin=400, fmax=1600):
    n_fft = 2048
    X = librosa.stft(x, n_fft=n_fft)
    Y = librosa.stft(y, n_fft=n_fft)

    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    band = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(band):
        return 0.0

    phi_X = np.angle(X[band, :])
    phi_Y = np.angle(Y[band, :])

    min_frames = min(phi_X.shape[1], phi_Y.shape[1])
    if min_frames < 2:
        return 0.0

    phi_X = phi_X[:, :min_frames]
    phi_Y = phi_Y[:, :min_frames]

    plv = np.abs(np.mean(np.exp(1j * (phi_X - phi_Y)), axis=1))
    return float(np.mean(plv))

def crackle_similarity(x, y, frame_length=512, hop_length=128):
    try:
        if len(x) < frame_length or len(y) < frame_length:
            return 0.0

        X_frames = librosa.util.frame(x, frame_length=frame_length, hop_length=hop_length)
        Y_frames = librosa.util.frame(y, frame_length=frame_length, hop_length=hop_length)

        kx = kurtosis(X_frames, axis=0, fisher=False, nan_policy='omit')
        ky = kurtosis(Y_frames, axis=0, fisher=False, nan_policy='omit')

        kx = np.nan_to_num(kx)
        ky = np.nan_to_num(ky)

        minlen = min(len(kx), len(ky))
        if minlen == 0:
            return 0.0

        d = np.mean(np.abs(kx[:minlen] - ky[:minlen]))
        return normalize_score(d, ref=10.0)
    except Exception:
        return 0.0

def band_similarity(x, y, sr=22050):
    bands = [(100, 300), (300, 800), (800, 3000)]

    Sx = np.abs(librosa.stft(x))
    Sy = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)

    scores = []
    for fmin, fmax in bands:
        band_mask = (freqs >= fmin) & (freqs <= fmax)
        if not np.any(band_mask):
            continue

        Px = np.mean(Sx[band_mask, :], axis=0)
        Py = np.mean(Sy[band_mask, :], axis=0)

        minlen = min(len(Px), len(Py))
        if minlen == 0:
            continue

        diff = np.sum(np.abs(Px[:minlen] - Py[:minlen]))
        denom = np.sum(Px[:minlen]) + 1e-8
        s = max(0.0, 1.0 - (diff / denom))
        scores.append(s)

    return float(np.mean(scores)) if scores else 0.0

def calculate_prism(original_path, synth_path, sr=22050):
    try:
        x, _ = librosa.load(original_path, sr=sr)
        y, _ = librosa.load(synth_path, sr=sr)

        if len(x) == 0 or len(y) == 0:
            return 0.0

        x = librosa.util.normalize(x)
        y = librosa.util.normalize(y)

        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]

        if min_len < 512:
            return 0.0

        s1 = traj_similarity(x, y, sr)
        s2 = cycle_similarity(x, y, sr)
        s3 = wheeze_similarity(x, y, sr)
        s4 = crackle_similarity(x, y)
        s5 = band_similarity(x, y, sr)

        score = (0.3 * s1) + (0.2 * s2) + (0.2 * s3) + (0.15 * s4) + (0.15 * s5)
        return float(score)

    except Exception as e:
        print(f"PRISM error for {os.path.basename(original_path)}: {e}")
        return 0.0

# =========================================================
# EVALUATOR
# =========================================================
class RespiratoryEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing on: {self.device}")

        self.frechet = FrechetAudioDistance(
            model_name="vggish",
            use_pca=False,
            use_activation=False,
            verbose=False
        )

        self.at_model = AudioTagging(checkpoint_path=None, device=self.device)
        self.sr_panns = 32000

    def canonical_name(self, filename):
        """
        Convert filenames like:
        xxx.wav
        into a common matching key.
        """
        name, ext = os.path.splitext(filename)
        lower_name = name.lower()

        suffixes_to_strip = [
            "_generated",
            "_gen",
            "_synth",
            "_synthetic",
        ]

        for suffix in suffixes_to_strip:
            if lower_name.endswith(suffix):
                name = name[: -len(suffix)]
                break

        return name.lower()

    def get_matched_file_pairs(self, original_path, generated_path):
        original_files = sorted(glob.glob(os.path.join(original_path, "*.wav")))
        generated_files = sorted(glob.glob(os.path.join(generated_path, "*.wav")))

        original_map = {}
        for f in original_files:
            base = os.path.basename(f)
            key = self.canonical_name(base)
            original_map[key] = f

        generated_map = {}
        for f in generated_files:
            base = os.path.basename(f)
            key = self.canonical_name(base)
            generated_map[key] = f

        common_keys = sorted(set(original_map.keys()).intersection(generated_map.keys()))

        print(f"Original files: {len(original_files)}")
        print(f"Generated files: {len(generated_files)}")
        print(f"Matched files: {len(common_keys)}")

        missing_in_generated = sorted(set(original_map.keys()) - set(generated_map.keys()))
        missing_in_original = sorted(set(generated_map.keys()) - set(original_map.keys()))

        if missing_in_generated:
            print(f"First few missing in generated folder: {missing_in_generated[:5]}")
        if missing_in_original:
            print(f"First few missing in original folder: {missing_in_original[:5]}")

        pairs = [(original_map[k], generated_map[k]) for k in common_keys]
        return pairs

    def extract_probs(self, file_paths):
        preds = []

        for path in file_paths:
            y, _ = librosa.load(path, sr=self.sr_panns, mono=True)

            if len(y) == 0:
                continue

            y = y[None, :]
            with torch.no_grad():
                _, clipwise = self.at_model.inference(y)
                preds.append(clipwise)

        if len(preds) == 0:
            return None

        raw = np.concatenate(preds, axis=0)
        raw = raw / (raw.sum(axis=1, keepdims=True) + 1e-12)
        return raw

    def _pad_audio_for_fad_pairs(self, pairs, temp_original, temp_generated, min_duration=1.0):
        for original_file, generated_file in pairs:
            original_name = os.path.basename(original_file)
            y_original, sr_original = librosa.load(original_file, sr=None, mono=True)
            y_generated, sr_generated = librosa.load(generated_file, sr=None, mono=True)

            min_samples_original = int(min_duration * sr_original)
            min_samples_generated = int(min_duration * sr_generated)

            if len(y_original) < min_samples_original:
                y_original = np.pad(y_original, (0, min_samples_original - len(y_original)), mode="constant")
            if len(y_generated) < min_samples_generated:
                y_generated = np.pad(y_generated, (0, min_samples_generated - len(y_generated)), mode="constant")

            # Save with identical filenames in temp folders so FAD compares aligned sets
            key_name = self.canonical_name(original_name) + ".wav"
            sf.write(os.path.join(temp_original, key_name), y_original, sr_original)
            sf.write(os.path.join(temp_generated, key_name), y_generated, sr_generated)

    def run(self):
        results = []

        for name, original_path, generated_path in EVAL_PAIRS:
            print("\n" + "=" * 70)
            print(f"Evaluating: {name}")
            print("=" * 70)

            pairs = self.get_matched_file_pairs(original_path, generated_path)
            if not pairs:
                print("No matching files found, skipping.")
                continue

            original_paths = [original for original, _ in pairs]
            generated_paths = [generated for _, generated in pairs]

            # -------------------------------------------------
            # Extract probabilities for original and generated
            # -------------------------------------------------
            original_probs = self.extract_probs(original_paths)
            generated_probs = self.extract_probs(generated_paths)

            if original_probs is None or generated_probs is None:
                print("Could not extract probabilities, skipping.")
                continue

            # -------------------------------------------------
            # FAD
            # -------------------------------------------------
            try:
                with tempfile.TemporaryDirectory() as temp_original, tempfile.TemporaryDirectory() as temp_generated:
                    self._pad_audio_for_fad_pairs(pairs, temp_original, temp_generated)
                    fad = self.frechet.score(temp_original, temp_generated)
            except Exception as e:
                print(f"[FAD Error] {e}")
                fad = -1.0

            # -------------------------------------------------
            # IS
            # -------------------------------------------------
            p_y = np.mean(generated_probs, axis=0, keepdims=True)
            kl_per_sample = generated_probs * (np.log(generated_probs + 1e-10) - np.log(p_y + 1e-10))
            is_score = float(np.exp(np.mean(np.sum(kl_per_sample, axis=1))))

            # -------------------------------------------------
            # KL
            # -------------------------------------------------
            P = np.mean(original_probs, axis=0) + 1e-10
            Q = np.mean(generated_probs, axis=0) + 1e-10
            P /= P.sum()
            Q /= Q.sum()
            kl_score = float(np.sum(P * np.log(P / Q)))

            # -------------------------------------------------
            # PRISM
            # -------------------------------------------------
            print("Calculating PRISM...")
            prism_vals = [
                calculate_prism(original_file, generated_file)
                for original_file, generated_file in pairs
            ]
            prism_score = float(np.mean(prism_vals)) if len(prism_vals) else 0.0

            results.append({
                "Model": name,
                "N files": len(pairs),
                "FAD ↓": fad,
                "KL ↓": kl_score,
                "IS ↑": is_score,
                "PRISM ↑": prism_score
            })

        return pd.DataFrame(results)

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    ev = RespiratoryEvaluator()
    df = ev.run()

    print("\n\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    if len(df) > 0:
        print(df.round(4).to_markdown(index=False))
    else:
        print("No results generated.")